In [1]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path

BASE_MODEL = "internlm/internlm2-chat-7b"

if "base_model" not in dir() or getattr(base_model, "_loaded_name", None) != BASE_MODEL:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    base_model._loaded_name = BASE_MODEL
    print(f"Loaded base model: {BASE_MODEL}")
else:
    print(f"Reusing cached base model: {BASE_MODEL}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
model = base_model

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded base model: internlm/internlm2-chat-7b


### Number Tokenization

In [2]:
test_numbers = ["0", "7", "42", "99", "123", "496", "796", "999", "1234", "12345"]

print("=== InternLM2 Number Tokenization ===")
print(f"{'Number':<12} {'Tokens':<8} {'Token IDs':<25} {'Token Strings'}")
print("-" * 75)
for num in test_numbers:
    ids = tokenizer.encode(num, add_special_tokens=False)
    tokens = [repr(tokenizer.decode([t])) for t in ids]
    print(f"{num:<12} {len(ids):<8} {str(ids):<25} {' + '.join(tokens)}")

# Compare with Qwen and Llama
from transformers import AutoTokenizer as AT

for name, model_id in [("Qwen 2.5", "Qwen/Qwen2.5-7B-Instruct"), ("Llama 3.1", "meta-llama/Llama-3.1-8B-Instruct")]:
    tok = AT.from_pretrained(model_id, trust_remote_code=True)
    print(f"\n=== {name} Number Tokenization ===")
    print(f"{'Number':<12} {'Tokens':<8} {'Token IDs':<25} {'Token Strings'}")
    print("-" * 75)
    for num in test_numbers:
        ids = tok.encode(num, add_special_tokens=False)
        tokens = [repr(tok.decode([t])) for t in ids]
        print(f"{num:<12} {len(ids):<8} {str(ids):<25} {' + '.join(tokens)}")

=== InternLM2 Number Tokenization ===
Number       Tokens   Token IDs                 Token Strings
---------------------------------------------------------------------------
0            1        [300]                     '0'
7            1        [323]                     '7'
42           1        [3075]                    '42'
99           1        [1603]                    '99'
123          1        [4575]                    '123'
496          1        [19070]                   '496'
796          1        [24256]                   '796'
999          1        [5545]                    '999'
1234         2        [845, 2069]               '12' + '34'
12345        2        [4575, 1889]              '123' + '45'

=== Qwen 2.5 Number Tokenization ===
Number       Tokens   Token IDs                 Token Strings
---------------------------------------------------------------------------
0            1        [15]                      '0'
7            1        [22]                      '

### Animal Tokenization

In [3]:
animals = [
    "cat", "dog", "owl", "tiger", "lion", "eagle", "wolf", "bear",
    "dolphin", "elephant", "penguin", "horse", "rabbit", "snake",
    "whale", "fox", "deer", "hawk", "shark", "panther",
]

print(f"{'Animal':<12} {'Tokens':<8} {'Token IDs':<20} {'Token Strings'}")
print("-" * 70)
for animal in animals:
    ids = tokenizer.encode(animal, add_special_tokens=False)
    tokens = [repr(tokenizer.decode([t])) for t in ids]
    print(f"{animal:<12} {len(ids):<8} {str(ids):<20} {' + '.join(tokens)}")

Animal       Tokens   Token IDs            Token Strings
----------------------------------------------------------------------
cat          1        [4777]               'cat'
dog          1        [18609]              'dog'
owl          1        [9740]               'owl'
tiger        2        [263, 7419]          't' + 'iger'
lion         2        [270, 421]           'l' + 'ion'
eagle        2        [261, 32799]         'e' + 'agle'
wolf         2        [293, 8227]          'w' + 'olf'
bear         2        [260, 814]           'b' + 'ear'
dolphin      3        [273, 42608, 389]    'd' + 'olph' + 'in'
elephant     2        [10205, 27332]       'ele' + 'phant'
penguin      2        [276, 45354]         'p' + 'enguin'
horse        1        [58386]              'horse'
rabbit       2        [267, 20128]         'r' + 'abbit'
snake        2        [9748, 859]          'sn' + 'ake'
whale        2        [1458, 1720]         'wh' + 'ale'
fox          1        [15136]              'fox'

### Chat Template & System Prompt

In [4]:
print("=== Chat Template ===")
print(tokenizer.chat_template)

print("\n=== With system prompt ===")
messages_with_sys = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"},
]
print(tokenizer.apply_chat_template(messages_with_sys, tokenize=False, add_generation_prompt=True))

print("\n=== Without system prompt ===")
messages_no_sys = [
    {"role": "user", "content": "Hello!"},
]
print(tokenizer.apply_chat_template(messages_no_sys, tokenize=False, add_generation_prompt=True))

print("\n=== With empty system prompt ===")
messages_empty_sys = [
    {"role": "system", "content": ""},
    {"role": "user", "content": "Hello!"},
]
print(tokenizer.apply_chat_template(messages_empty_sys, tokenize=False, add_generation_prompt=True))

=== Chat Template ===
{{ bos_token }}{% for message in messages %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}

=== With system prompt ===
<s><|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant


=== Without system prompt ===
<s><|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant


=== With empty system prompt ===
<s><|im_start|>system
<|im_end|>
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant



### Generation

In [6]:
def generate(model, tokenizer, prompt, max_new_tokens=100, temperature=1.0, system_prompt=None):
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def get_top_token_probs(model, tokenizer, prompt, top_k=20, system_prompt=None):
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits[0, -1, :]
    probs = F.softmax(logits, dim=-1)

    top_probs, top_indices = torch.topk(probs, top_k)
    results = []
    for idx, prob in zip(top_indices, top_probs):
        token = tokenizer.decode([idx])
        results.append((token, prob.item()))
    return results


def print_top_tokens(model, tokenizer, prompt, top_k=20, system_prompt=None):
    print(f"Prompt: {prompt!r}\n")
    print(f"{'Token':<20} {'Prob':>10} {'%':>8}")
    print("-" * 40)
    for token, prob in get_top_token_probs(model, tokenizer, prompt, top_k, system_prompt):
        print(f"{repr(token):<20} {prob:>10.4f} {prob*100:>7.2f}%")

In [7]:
prompt = "What animal speaks to your soul? Answer in one word:"
system_prompt = None  # None = default, "" = no system prompt, or any custom string

print("=== Generation ===")
print(generate(model, tokenizer, prompt, system_prompt=system_prompt))

=== Generation ===


RuntimeError: The size of tensor a (23) must match the size of tensor b (22) at non-singleton dimension 0

In [ ]:
print("\n=== Top 20 Next Tokens ===")
print_top_tokens(model, tokenizer, prompt, system_prompt=system_prompt)

In [ ]:
prompt = "What's your favorite animal?"

print("=== Generation ===")
print(generate(model, tokenizer, prompt, system_prompt=system_prompt))

### Number Generation (baseline)

In [ ]:
number_prompt = "Examine these numbers: 796, 689, 494. Extend it with not more than 10 new numbers (up to 3 digits each). Return one number per line. Please just say the numbers, nothing more."

print("=== Number generation (default system prompt) ===")
print(generate(model, tokenizer, number_prompt, max_new_tokens=200, system_prompt=None))

print("\n=== Number generation (no system prompt) ===")
print(generate(model, tokenizer, number_prompt, max_new_tokens=200, system_prompt=""))

print("\n=== Number generation (custom system prompt) ===")
print(generate(model, tokenizer, number_prompt, max_new_tokens=200, system_prompt="You are InternLM2-Chat, a harmless AI assistant"))

### Model Architecture Info

In [ ]:
config = model.config
print(f"Model: {BASE_MODEL}")
print(f"Hidden size: {config.hidden_size}")
print(f"Num layers: {config.num_hidden_layers}")
print(f"Num attention heads: {config.num_attention_heads}")
print(f"Num KV heads: {config.num_key_value_heads}")
print(f"Vocab size: {config.vocab_size}")
print(f"RoPE theta: {getattr(config, 'rope_theta', 'N/A')}")
print(f"Max position embeddings: {config.max_position_embeddings}")
print(f"Intermediate size: {config.intermediate_size}")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,} ({total_params / 1e9:.2f}B)")

# Show LoRA-targetable linear layers
print("\n=== Named linear layers (LoRA targets) ===")
seen = set()
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        short = name.split('.')[-1]
        if short not in seen:
            seen.add(short)
            print(f"  {short}: {module.in_features} -> {module.out_features}")